In [2]:

%pip install pygame

import pygame
import random
import sys

pygame.init()

WIDTH, HEIGHT = 960, 600
FPS = 60
WORLD_WIDTH = 3600
GRAVITY = 0.65
JUMP_POWER = -13
PLAYER_SPEED = 5

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Pixel Quest: Human vs AI")
clock = pygame.time.Clock()

FONT = pygame.font.SysFont("consolas", 18, bold=True)
BIG_FONT = pygame.font.SysFont("consolas", 38, bold=True)

SKY = (100, 190, 255)
WHITE = (255, 255, 255)
BLACK = (30, 30, 40)
GREEN = (60, 180, 75)
DARK_GREEN = (35, 115, 50)
BROWN = (130, 75, 40)
GOLD = (255, 210, 40)
RED = (230, 55, 65)
PURPLE = (170, 70, 220)


platforms = [
    pygame.Rect(0, 520, 600, 80),
    pygame.Rect(700, 520, 550, 80),
    pygame.Rect(1370, 520, 600, 80),
    pygame.Rect(2090, 520, 580, 80),
    pygame.Rect(2780, 520, 820, 80),
    pygame.Rect(200, 420, 150, 25),
    pygame.Rect(420, 340, 150, 25),
    pygame.Rect(780, 420, 180, 25),
    pygame.Rect(1030, 350, 140, 25),
    pygame.Rect(1480, 420, 170, 25),
    pygame.Rect(1730, 340, 160, 25),
    pygame.Rect(2180, 420, 170, 25),
    pygame.Rect(2420, 350, 160, 25),
    pygame.Rect(2920, 420, 160, 25),
    pygame.Rect(3180, 350, 180, 25),
]

finish_x = 3470


def make_player(x, color, name):
    return {
        "rect": pygame.Rect(x, 450, 30, 40),
        "vx": 0.0,
        "vy": 0.0,
        "color": color,
        "name": name,
        "grounded": False,
        "lives": 3,
        "score": 0,
        "invulnerable": 0,
        "finished": False,
        "respawn": 0,
    }

def reset_character(c, x=70):
    c["rect"].x = x
    c["rect"].y = 400
    c["vx"] = 0
    c["vy"] = 0
    c["grounded"] = False
    c["invulnerable"] = 100
    c["respawn"] = 0

player = make_player(70, RED, "YOU")
rival = make_player(110, PURPLE, "AI")

# ---------------- COINS ----------------

coins = []

for x in range(150, WORLD_WIDTH - 100, 110):
    coins.append(
        pygame.Rect(x, random.choice([385, 445, 475]), 18, 18)
    )

for x, y in [
    (240, 375), (285, 375), (460, 295), (510, 295),
    (820, 375), (870, 375), (1060, 305), (1110, 305),
    (1520, 375), (1770, 295), (2220, 375), (2470, 305),
    (2960, 375), (3220, 305)
]:
    coins.append(pygame.Rect(x, y, 18, 18))

# ---------------- ENEMIES ----------------

enemies = []

for x in [360, 850, 1130, 1550, 1840, 2280, 2500, 3000, 3300]:
    enemies.append({
        "rect": pygame.Rect(x, 480, 30, 35),
        "vx": random.choice([-1.5, 1.5]),
        "alive": True,
        "left": x - 75,
        "right": x + 75,
    })

# ---------------- DRAWING ----------------

def draw_text(text, x, y, color=WHITE, font=FONT):
    shadow = font.render(text, True, BLACK)
    image = font.render(text, True, color)
    screen.blit(shadow, (x + 2, y + 2))
    screen.blit(image, (x, y))

def draw_background(camera):
    screen.fill(SKY)

    # Pixel clouds
    for i in range(12):
        x = (i * 350 - int(camera * 0.3)) % (WIDTH + 180) - 90
        y = 65 + (i % 3) * 45
        pygame.draw.rect(screen, WHITE, (x, y, 70, 22))
        pygame.draw.rect(screen, WHITE, (x + 15, y - 12, 35, 35))
        pygame.draw.rect(screen, WHITE, (x + 42, y - 5, 30, 27))

    # Hills
    for i in range(10):
        x = i * 430 - int(camera * 0.5) % 430
        pygame.draw.ellipse(
            screen, (80, 200, 115), (x, 390, 270, 160)
        )

def draw_world(camera):
    draw_background(camera)

    for p in platforms:
        r = pygame.Rect(p.x - camera, p.y, p.width, p.height)

        if r.right < 0 or r.left > WIDTH:
            continue

        pygame.draw.rect(screen, BROWN, r)
        pygame.draw.rect(
            screen, GREEN,
            (r.x, r.y, r.width, min(10, r.height))
        )

        for x in range(r.left, r.right, 40):
            pygame.draw.rect(
                screen, DARK_GREEN, (x, r.y, 4, 5)
            )

    # Coins
    for coin in coins:
        r = pygame.Rect(
            coin.x - camera, coin.y, coin.width, coin.height
        )

        if -30 < r.x < WIDTH + 30:
            pygame.draw.ellipse(screen, GOLD, r)
            pygame.draw.rect(
                screen, (255, 245, 130),
                (r.x + 6, r.y + 3, 5, 12)
            )

    # Enemies
    for e in enemies:
        if not e["alive"]:
            continue

        r = e["rect"].move(-camera, 0)

        if -50 < r.x < WIDTH + 50:
            pygame.draw.rect(screen, (130, 65, 35), r)
            pygame.draw.rect(
                screen, WHITE, (r.x + 5, r.y + 7, 7, 8)
            )
            pygame.draw.rect(
                screen, WHITE, (r.x + 18, r.y + 7, 7, 8)
            )
            pygame.draw.rect(
                screen, BLACK, (r.x + 8, r.y + 10, 3, 4)
            )
            pygame.draw.rect(
                screen, BLACK, (r.x + 21, r.y + 10, 3, 4)
            )

    # Finish flag
    fx = finish_x - camera
    pygame.draw.rect(screen, BROWN, (fx, 350, 8, 170))
    pygame.draw.rect(screen, GREEN, (fx + 8, 350, 75, 40))
    pygame.draw.rect(screen, WHITE, (fx + 8, 390, 75, 40))
    draw_text("FINISH", fx - 30, 320, GOLD)

def draw_character(c, camera):
    if c["invulnerable"] > 0 and c["invulnerable"] % 10 < 5:
        return

    r = c["rect"].move(-camera, 0)
    color = c["color"]

    # Body
    pygame.draw.rect(screen, color, (r.x + 4, r.y + 12, 22, 18))
    pygame.draw.rect(screen, color, (r.x + 1, r.y + 28, 12, 12))
    pygame.draw.rect(screen, color, (r.x + 17, r.y + 28, 12, 12))

    # Hat and face
    pygame.draw.rect(screen, color, (r.x + 3, r.y, 24, 9))
    pygame.draw.rect(
        screen, (255, 205, 155), (r.x + 6, r.y + 8, 19, 15)
    )
    pygame.draw.rect(screen, BLACK, (r.x + 19, r.y + 12, 3, 4))

    # Shoes
    pygame.draw.rect(screen, BLACK, (r.x, r.y + 36, 14, 4))
    pygame.draw.rect(screen, BLACK, (r.x + 16, r.y + 36, 14, 4))

# ---------------- PHYSICS ----------------

def move_character(c):
    c["rect"].x += round(c["vx"])

    for p in platforms:
        if c["rect"].colliderect(p):
            if c["vx"] > 0:
                c["rect"].right = p.left
            elif c["vx"] < 0:
                c["rect"].left = p.right
            c["vx"] = 0

    c["vy"] = min(c["vy"] + GRAVITY, 15)
    c["rect"].y += round(c["vy"])
    c["grounded"] = False

    for p in platforms:
        if c["rect"].colliderect(p):
            if c["vy"] > 0:
                c["rect"].bottom = p.top
                c["grounded"] = True
            elif c["vy"] < 0:
                c["rect"].top = p.bottom
            c["vy"] = 0

    if c["rect"].top > HEIGHT + 100:
        c["lives"] -= 1
        reset_character(c)

# ---------------- AI ----------------

def ai_decision():
    if rival["finished"] or rival["respawn"] > 0:
        rival["vx"] = 0
        return

    rival["vx"] = 3.4
    r = rival["rect"]
    next_x = r.right + 45

    # Detect a gap in the ground.
    ground_ahead = any(
        p.top >= 500 and p.left <= next_x < p.right
        for p in platforms
    )

    # Detect enemies ahead.
    enemy_ahead = any(
        e["alive"]
        and 0 <= e["rect"].centerx - r.centerx < 85
        and abs(e["rect"].bottom - r.bottom) < 70
        for e in enemies
    )

    # Detect elevated platforms.
    elevated_ahead = any(
        0 < p.centerx - r.centerx < 100
        and p.top < r.bottom - 35
        and p.top > r.bottom - 160
        for p in platforms
    )

    if rival["grounded"] and (
        enemy_ahead or not ground_ahead or elevated_ahead
    ):
        rival["vy"] = JUMP_POWER
        rival["grounded"] = False

# ---------------- MAIN GAME ----------------

camera = 0
game_state = "playing"
message = ""

print("PIXEL QUEST started!")
print("Move: A/D or arrow keys | Jump: SPACE, W or UP")
print("Restart after the game: R | Quit: ESC")

running = True

while running:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if event.key == pygame.K_r and game_state != "playing":
                # Restart the complete notebook game.
                player = make_player(70, RED, "YOU")
                rival = make_player(110, PURPLE, "AI")
                coins = []

                for x in range(150, WORLD_WIDTH - 100, 110):
                    coins.append(
                        pygame.Rect(
                            x, random.choice([385, 445, 475]), 18, 18
                        )
                    )

                for x, y in [
                    (240, 375), (285, 375), (460, 295), (510, 295),
                    (820, 375), (870, 375), (1060, 305), (1110, 305),
                    (1520, 375), (1770, 295), (2220, 375),
                    (2470, 305), (2960, 375), (3220, 305)
                ]:
                    coins.append(pygame.Rect(x, y, 18, 18))

                enemies = []
                for x in [
                    360, 850, 1130, 1550, 1840,
                    2280, 2500, 3000, 3300
                ]:
                    enemies.append({
                        "rect": pygame.Rect(x, 480, 30, 35),
                        "vx": random.choice([-1.5, 1.5]),
                        "alive": True,
                        "left": x - 75,
                        "right": x + 75,
                    })

                camera = 0
                game_state = "playing"
                message = ""

    if game_state == "playing":
        keys = pygame.key.get_pressed()

        # Human controls
        player["vx"] = 0

        if keys[pygame.K_LEFT] or keys[pygame.K_a]:
            player["vx"] = -PLAYER_SPEED

        if keys[pygame.K_RIGHT] or keys[pygame.K_d]:
            player["vx"] = PLAYER_SPEED

        if (
            keys[pygame.K_SPACE]
            or keys[pygame.K_UP]
            or keys[pygame.K_w]
        ) and player["grounded"]:
            player["vy"] = JUMP_POWER
            player["grounded"] = False

        # AI makes its decision.
        ai_decision()

        # Update both competitors.
        for c in [player, rival]:
            if c["respawn"] > 0:
                c["respawn"] -= 1
                continue

            move_character(c)

            if c["invulnerable"] > 0:
                c["invulnerable"] -= 1

            # Collect coins.
            for coin in coins[:]:
                if c["rect"].colliderect(coin):
                    coins.remove(coin)
                    c["score"] += 10

            # Enemy collisions.
            for e in enemies:
                if not e["alive"]:
                    continue

                if c["rect"].colliderect(e["rect"]):
                    if (
                        c["vy"] > 0
                        and c["rect"].bottom - e["rect"].top < 22
                    ):
                        e["alive"] = False
                        c["vy"] = -8
                        c["score"] += 25

                    elif c["invulnerable"] == 0:
                        c["lives"] -= 1
                        c["invulnerable"] = 90
                        c["vx"] = (
                            -4 if c["rect"].centerx > e["rect"].centerx
                            else 4
                        )
                        c["vy"] = -7

            # Finish line.
            if c["rect"].centerx >= finish_x:
                c["finished"] = True

        # Move enemies.
        for e in enemies:
            if not e["alive"]:
                continue

            e["rect"].x += round(e["vx"])

            if (
                e["rect"].x <= e["left"]
                or e["rect"].x >= e["right"]
            ):
                e["vx"] *= -1

        # Follow the human with the camera.
        camera = max(
            0,
            min(
                WORLD_WIDTH - WIDTH,
                player["rect"].centerx - WIDTH // 3
            )
        )

        # End conditions.
        if player["finished"] and rival["finished"]:
            game_state = "over"
            message = "YOU WIN!" if player["rect"].centerx >= rival["rect"].centerx else "AI WINS!"
        elif player["finished"]:
            game_state = "over"
            message = "YOU WIN! FINISH LINE REACHED!"
        elif rival["finished"]:
            game_state = "over"
            message = "AI WINS! THE RIVAL BEAT YOU!"
        elif player["lives"] <= 0:
            game_state = "over"
            message = "GAME OVER! AI SURVIVED."
        elif rival["lives"] <= 0:
            game_state = "over"
            message = "YOU WIN! AI RAN OUT OF LIVES."

    # Draw the game.
    draw_world(camera)
    draw_character(player, camera)
    draw_character(rival, camera)

    # Heads-up display.
    pygame.draw.rect(screen, BLACK, (10, 10, 375, 88))

    draw_text(
        f"YOU HP:{player['lives']} COINS:{player['score']}",
        20, 18, (255, 110, 110)
    )
    draw_text(
        f"AI  HP:{rival['lives']} COINS:{rival['score']}",
        20, 48, (210, 150, 255)
    )

    draw_text(
        "MOVE: A/D or ARROWS | JUMP: SPACE/W/UP",
        15, HEIGHT - 30
    )


    bx, by, bw = 610, 18, 325
    pygame.draw.rect(screen, BLACK, (bx, by, bw, 45))
    pygame.draw.rect(
        screen, WHITE, (bx + 10, by + 20, bw - 20, 8)
    )

    for c, color, offset in [
        (player, RED, 14),
        (rival, PURPLE, 25)
    ]:
        progress = min(1, c["rect"].centerx / finish_x)
        marker_x = bx + 10 + int((bw - 20) * progress)
        pygame.draw.rect(
            screen, color, (marker_x - 4, by + offset, 8, 8)
        )

    if game_state == "over":
        overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))

        title = BIG_FONT.render(message, True, GOLD)
        screen.blit(
            title,
            title.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 30))
        )

        restart = FONT.render(
            "Press R to play again | ESC to quit",
            True, WHITE
        )
        screen.blit(
            restart,
            restart.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 30))
        )

    pygame.display.flip()

pygame.quit()
print("Pixel Quest closed.")





Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 11.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.
pygame 2.6.1 (SDL 2.28.4, Python 3.9.18)
Hello from the pygame community. https://www.pygame.org/contribute.html
PIXEL QUEST started!
Move: A/D or arrow keys | Jump: SPACE, W or UP
Restart after the game: R | Quit: ESC
Pixel Quest closed.


In [3]:

%pip install pygame

import pygame
import random
import sys


pygame.init()

WIDTH, HEIGHT = 960, 600
FPS = 60
WORLD_WIDTH = 3600
GRAVITY = 0.65
JUMP_POWER = -13
PLAYER_SPEED = 5
FINISH_X = 3470

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Pixel Quest: Human vs Smart AI")
clock = pygame.time.Clock()

FONT = pygame.font.SysFont("consolas", 18, bold=True)
BIG_FONT = pygame.font.SysFont("consolas", 36, bold=True)

SKY = (100, 190, 255)
WHITE = (255, 255, 255)
BLACK = (30, 30, 40)
GREEN = (60, 180, 75)
DARK_GREEN = (35, 115, 50)
BROWN = (130, 75, 40)
GOLD = (255, 210, 40)
RED = (230, 55, 65)
PURPLE = (170, 70, 220)

# ---------------- WORLD ----------------

platforms = [
    pygame.Rect(0, 520, 600, 80),
    pygame.Rect(700, 520, 550, 80),
    pygame.Rect(1370, 520, 600, 80),
    pygame.Rect(2090, 520, 580, 80),
    pygame.Rect(2780, 520, 820, 80),

    pygame.Rect(200, 420, 150, 25),
    pygame.Rect(420, 340, 150, 25),
    pygame.Rect(780, 420, 180, 25),
    pygame.Rect(1030, 350, 140, 25),
    pygame.Rect(1480, 420, 170, 25),
    pygame.Rect(1730, 340, 160, 25),
    pygame.Rect(2180, 420, 170, 25),
    pygame.Rect(2420, 350, 160, 25),
    pygame.Rect(2920, 420, 160, 25),
    pygame.Rect(3180, 350, 180, 25),
]

# ---------------- CHARACTERS ----------------

def make_character(x, color, name):
    return {
        "rect": pygame.Rect(x, 450, 30, 40),
        "vx": 0.0,
        "vy": 0.0,
        "color": color,
        "name": name,
        "grounded": False,
        "lives": 3,
        "score": 0,
        "invulnerable": 0,
        "finished": False,
        "dead": False,
    }

player = make_character(70, RED, "YOU")
rival = make_character(110, PURPLE, "AI")

# ---------------- COINS ----------------

def make_coins():
    result = []

    for x in range(150, WORLD_WIDTH - 100, 110):
        result.append(
            pygame.Rect(x, random.choice([385, 445, 475]), 18, 18)
        )

    for x, y in [
        (240, 375), (285, 375), (460, 295), (510, 295),
        (820, 375), (870, 375), (1060, 305), (1110, 305),
        (1520, 375), (1770, 295), (2220, 375), (2470, 305),
        (2960, 375), (3220, 305)
    ]:
        result.append(pygame.Rect(x, y, 18, 18))

    return result

coins = make_coins()

# ---------------- ENEMIES ----------------

def make_enemies():
    result = []

    for x in [360, 850, 1130, 1550, 1840, 2280, 2500, 3000, 3300]:
        result.append({
            "rect": pygame.Rect(x, 480, 30, 35),
            "vx": random.choice([-1.5, 1.5]),
            "alive": True,
            "left": x - 75,
            "right": x + 75,
        })

    return result

enemies = make_enemies()

# ---------------- AI MEMORY ----------------

ai_memory = {
    "last_x": None,
    "stuck_frames": 0,
    "jump_cooldown": 0,
}

camera = 0
game_state = "playing"
message = ""

# ---------------- DRAWING ----------------

def draw_text(text, x, y, color=WHITE, font=FONT):
    screen.blit(font.render(text, True, BLACK), (x + 2, y + 2))
    screen.blit(font.render(text, True, color), (x, y))

def draw_background():
    screen.fill(SKY)

    # Clouds
    for i in range(12):
        x = (i * 350 - int(camera * 0.3)) % (WIDTH + 180) - 90
        y = 65 + (i % 3) * 45

        pygame.draw.rect(screen, WHITE, (x, y, 70, 22))
        pygame.draw.rect(screen, WHITE, (x + 15, y - 12, 35, 35))
        pygame.draw.rect(screen, WHITE, (x + 42, y - 5, 30, 27))

    # Distant hills
    for i in range(10):
        x = i * 430 - int(camera * 0.5) % 430
        pygame.draw.ellipse(
            screen, (80, 200, 115), (x, 390, 270, 160)
        )

def draw_character(c):
    if c["invulnerable"] > 0 and c["invulnerable"] % 10 < 5:
        return

    r = c["rect"].move(-camera, 0)
    color = c["color"]

    # Pixel body
    pygame.draw.rect(screen, color, (r.x + 4, r.y + 12, 22, 18))
    pygame.draw.rect(screen, color, (r.x + 1, r.y + 28, 12, 12))
    pygame.draw.rect(screen, color, (r.x + 17, r.y + 28, 12, 12))

    # Hat and face
    pygame.draw.rect(screen, color, (r.x + 3, r.y, 24, 9))
    pygame.draw.rect(
        screen, (255, 205, 155), (r.x + 6, r.y + 8, 19, 15)
    )
    pygame.draw.rect(screen, BLACK, (r.x + 19, r.y + 12, 3, 4))

    # Shoes
    pygame.draw.rect(screen, BLACK, (r.x, r.y + 36, 14, 4))
    pygame.draw.rect(screen, BLACK, (r.x + 16, r.y + 36, 14, 4))

def draw_world():
    draw_background()

    # Platforms
    for p in platforms:
        r = pygame.Rect(p.x - camera, p.y, p.width, p.height)

        if r.right < 0 or r.left > WIDTH:
            continue

        pygame.draw.rect(screen, BROWN, r)
        pygame.draw.rect(
            screen, GREEN, (r.x, r.y, r.width, min(10, r.height))
        )

        for x in range(r.left, r.right, 40):
            pygame.draw.rect(screen, DARK_GREEN, (x, r.y, 4, 5))

    # Coins
    for coin in coins:
        x = coin.x - camera

        if -30 < x < WIDTH + 30:
            pygame.draw.ellipse(
                screen, GOLD, (x, coin.y, coin.width, coin.height)
            )
            pygame.draw.rect(
                screen, (255, 245, 130),
                (x + 6, coin.y + 3, 5, 12)
            )

    # Enemies
    for e in enemies:
        if not e["alive"]:
            continue

        r = e["rect"].move(-camera, 0)

        if -50 < r.x < WIDTH + 50:
            pygame.draw.rect(screen, (130, 65, 35), r)
            pygame.draw.rect(screen, WHITE, (r.x + 5, r.y + 7, 7, 8))
            pygame.draw.rect(screen, WHITE, (r.x + 18, r.y + 7, 7, 8))
            pygame.draw.rect(screen, BLACK, (r.x + 8, r.y + 10, 3, 4))
            pygame.draw.rect(screen, BLACK, (r.x + 21, r.y + 10, 3, 4))

    # Finish flag
    fx = FINISH_X - camera
    pygame.draw.rect(screen, BROWN, (fx, 350, 8, 170))
    pygame.draw.rect(screen, GREEN, (fx + 8, 350, 75, 40))
    pygame.draw.rect(screen, WHITE, (fx + 8, 390, 75, 40))
    draw_text("FINISH", fx - 30, 320, GOLD)

    draw_character(player)
    draw_character(rival)

    # HUD
    pygame.draw.rect(screen, BLACK, (10, 10, 390, 90))
    draw_text(
        f"YOU  Lives:{player['lives']} Coins:{player['score']}",
        20, 18, (255, 120, 120)
    )
    draw_text(
        f"AI   Lives:{rival['lives']} Coins:{rival['score']}",
        20, 48, (210, 150, 255)
    )

    # Race progress bar
    bx, by, bw = 620, 18, 315
    pygame.draw.rect(screen, BLACK, (bx, by, bw, 48))
    pygame.draw.rect(screen, WHITE, (bx + 10, by + 20, bw - 20, 8))

    for c, color, offset in [
        (player, RED, 14),
        (rival, PURPLE, 27)
    ]:
        progress = max(0, min(1, c["rect"].centerx / FINISH_X))
        mx = bx + 10 + int((bw - 20) * progress)
        pygame.draw.rect(screen, color, (mx - 4, by + offset, 8, 8))

    draw_text(
        "MOVE: A/D or ARROWS | JUMP: SPACE/W/UP",
        15, HEIGHT - 30
    )

# ---------------- PHYSICS ----------------

def move_character(c):
    # Horizontal collision
    c["rect"].x += round(c["vx"])

    for p in platforms:
        if c["rect"].colliderect(p):
            if c["vx"] > 0:
                c["rect"].right = p.left
            elif c["vx"] < 0:
                c["rect"].left = p.right

            c["vx"] = 0

    # Vertical collision
    c["vy"] = min(c["vy"] + GRAVITY, 15)
    c["rect"].y += round(c["vy"])
    c["grounded"] = False

    for p in platforms:
        if c["rect"].colliderect(p):
            if c["vy"] > 0:
                c["rect"].bottom = p.top
                c["grounded"] = True
            elif c["vy"] < 0:
                c["rect"].top = p.bottom

            c["vy"] = 0

    # World boundaries
    c["rect"].left = max(0, c["rect"].left)
    c["rect"].right = min(WORLD_WIDTH, c["rect"].right)

    # Falling
    if c["rect"].top > HEIGHT + 100:
        c["lives"] -= 1
        c["rect"].x = 70
        c["rect"].y = 400
        c["vx"] = 0
        c["vy"] = 0
        c["invulnerable"] = 90

# ---------------- SMART AI ----------------

def ai_decision():
    c = rival
    r = c["rect"]

    if c["finished"]:
        c["vx"] = 0
        return

    # Detect whether the rival is stuck
    if ai_memory["last_x"] == r.x:
        ai_memory["stuck_frames"] += 1
    else:
        ai_memory["stuck_frames"] = 0

    ai_memory["last_x"] = r.x

    if ai_memory["jump_cooldown"] > 0:
        ai_memory["jump_cooldown"] -= 1

    # Normal forward movement
    c["vx"] = 3.2

    # Detect current platform
    current_ground = None

    if c["grounded"]:
        for p in platforms:
            if (
                abs(r.bottom - p.top) <= 3
                and r.right > p.left
                and r.left < p.right
            ):
                current_ground = p
                break

    # Find the next ground platform
    ground_platforms = sorted(
        [p for p in platforms if p.top >= 500],
        key=lambda p: p.left
    )

    next_ground = next(
        (p for p in ground_platforms if p.left > r.centerx + 5),
        None
    )

    should_jump = False

    # Gap detection
    if current_ground and next_ground and c["grounded"]:
        distance_to_edge = current_ground.right - r.right
        gap_width = next_ground.left - current_ground.right

        if 0 <= distance_to_edge <= 55 and 0 < gap_width <= 155:
            should_jump = True

        # Do not blindly run into gaps that appear too wide.
        elif 0 <= distance_to_edge <= 18 and gap_width > 155:
            c["vx"] = 0

    # Enemy detection
    enemy_ahead = any(
        e["alive"]
        and 0 <= e["rect"].left - r.right < 90
        and abs(e["rect"].bottom - r.bottom) < 75
        for e in enemies
    )

    if enemy_ahead and c["grounded"]:
        should_jump = True

    # Elevated platforms
    elevated_ahead = any(
        0 < p.left - r.right < 65
        and 45 <= r.bottom - p.top <= 125
        and p.top < 500
        for p in platforms
    )

    if elevated_ahead and c["grounded"]:
        should_jump = True

    # Recovery: try a jump if movement stalls.
    if ai_memory["stuck_frames"] > 20:
        if c["grounded"] and ai_memory["jump_cooldown"] == 0:
            c["vy"] = JUMP_POWER
            c["grounded"] = False
            ai_memory["jump_cooldown"] = 30

        c["vx"] = 2.5

    # Execute planned jump
    if (
        should_jump
        and c["grounded"]
        and ai_memory["jump_cooldown"] == 0
    ):
        c["vy"] = JUMP_POWER
        c["grounded"] = False
        ai_memory["jump_cooldown"] = 25

    # Never move beyond the finish line
    if r.centerx >= FINISH_X:
        c["finished"] = True
        c["vx"] = 0

# ---------------- RESET ----------------

def reset_game():
    global player, rival, coins, enemies, camera
    global game_state, message, ai_memory

    player = make_character(70, RED, "YOU")
    rival = make_character(110, PURPLE, "AI")

    coins = make_coins()
    enemies = make_enemies()
    camera = 0
    game_state = "playing"
    message = ""

    ai_memory = {
        "last_x": None,
        "stuck_frames": 0,
        "jump_cooldown": 0,
    }

# ---------------- MAIN LOOP ----------------

running = True

while running:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_ESCAPE:
                running = False

            if event.key == pygame.K_r and game_state != "playing":
                reset_game()

    if game_state == "playing":
        keys = pygame.key.get_pressed()

        # Human movement
        player["vx"] = 0

        if keys[pygame.K_LEFT] or keys[pygame.K_a]:
            player["vx"] = -PLAYER_SPEED

        if keys[pygame.K_RIGHT] or keys[pygame.K_d]:
            player["vx"] = PLAYER_SPEED

        if (
            keys[pygame.K_SPACE]
            or keys[pygame.K_UP]
            or keys[pygame.K_w]
        ) and player["grounded"]:
            player["vy"] = JUMP_POWER
            player["grounded"] = False

        # AI decision
        ai_decision()

        # Update both characters
        for c in (player, rival):
            move_character(c)

            if c["invulnerable"] > 0:
                c["invulnerable"] -= 1

            # Coin collection
            for coin in coins[:]:
                if c["rect"].colliderect(coin):
                    coins.remove(coin)
                    c["score"] += 10

            # Enemy collisions
            for e in enemies:
                if not e["alive"]:
                    continue

                if c["rect"].colliderect(e["rect"]):
                    if (
                        c["vy"] > 0
                        and c["rect"].bottom - e["rect"].top < 22
                    ):
                        e["alive"] = False
                        c["vy"] = -8
                        c["score"] += 25

                    elif c["invulnerable"] == 0:
                        c["lives"] -= 1
                        c["invulnerable"] = 90
                        c["vy"] = -7
                        c["vx"] = -4

            if c["rect"].centerx >= FINISH_X:
                c["finished"] = True

        # Move enemies
        for e in enemies:
            if e["alive"]:
                e["rect"].x += round(e["vx"])

                if (
                    e["rect"].x <= e["left"]
                    or e["rect"].x >= e["right"]
                ):
                    e["vx"] *= -1

        # Camera follows the player
        camera = max(
            0,
            min(
                WORLD_WIDTH - WIDTH,
                player["rect"].centerx - WIDTH // 3
            )
        )

        # End conditions
        if player["finished"]:
            game_state = "over"
            message = "YOU WIN! YOU REACHED THE FINISH!"

        elif rival["finished"]:
            game_state = "over"
            message = "AI WINS! THE RIVAL BEAT YOU!"

        elif player["lives"] <= 0:
            game_state = "over"
            message = "GAME OVER! YOU RAN OUT OF LIVES!"

        elif rival["lives"] <= 0:
            game_state = "over"
            message = "YOU WIN! THE AI RAN OUT OF LIVES!"

    # Draw current frame
    draw_world()

    if game_state == "over":
        overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))

        title = BIG_FONT.render(message, True, GOLD)
        screen.blit(
            title,
            title.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 25))
        )

        restart = FONT.render(
            "Press R to restart | ESC to quit",
            True, WHITE
        )
        screen.blit(
            restart,
            restart.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 30))
        )

    pygame.display.flip()

pygame.quit()
sys.exit()


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


SystemExit: 

/opt/intel/oneapi/intelpython/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
